In [ ]:
%run ./NB_Config_SELLING_SYSTEM

from datetime import datetime
import re
import uuid
from pyspark.sql import functions as F

PIPELINE_RUN_ID = str(uuid.uuid4())
start_time = datetime.utcnow()
results = []

def _schema_key(column_name):
    return re.sub(r'[^a-z0-9]', '', str(column_name).lower())

def _resolve(dataframe, column_name):
    if column_name in dataframe.columns:
        return column_name
    key = _schema_key(column_name)
    return next((name for name in dataframe.columns if _schema_key(name) == key), None)

def active_bronze(table_name):
    return (spark.read.format('delta').load(f'{BRONZE_LH_ABFSS}/{BRONZE_SCHEMA}/{table_name}')
                .filter(F.col('_IS_DELETED') == False))

def apply_rename_map(dataframe, rename_map):
    for source_column, silver_column in rename_map.items():
        if source_column in dataframe.columns and source_column != silver_column:
            if silver_column not in dataframe.columns:
                dataframe = dataframe.withColumnRenamed(source_column, silver_column)
    return dataframe

def user_lookup():
    users = active_bronze('SF_User')
    user_id = _resolve(users, 'UserId')
    full_name = _resolve(users, 'FullName')
    manager_id = _resolve(users, 'ManagerId')
    profile_name = _resolve(users, 'ProfileName')
    selected = [F.col(user_id).alias('__user_id'), F.col(full_name).alias('__user_name')]
    if manager_id:
        selected.append(F.col(manager_id).alias('__manager_id'))
    if profile_name:
        selected.append(F.col(profile_name).alias('__profile_name'))
    return users.select(*selected).dropDuplicates(['__user_id'])

def apply_user_rules(dataframe):
    federation_id = _resolve(dataframe, 'FederationIdentifier')
    if federation_id:
        dataframe = dataframe.withColumn('FederationIdentifier-Short', F.split(F.col(federation_id), '@').getItem(0))
    user_id = _resolve(dataframe, 'UserId')
    manager_id = _resolve(dataframe, 'ManagerId')
    if user_id and manager_id:
        managers = user_lookup().select(
            F.col('__user_id').alias('__manager_key'),
            F.col('__user_name').alias('ManagerFullName'))
        dataframe = dataframe.join(managers, F.col(manager_id) == F.col('__manager_key'), 'left').drop('__manager_key')
    if _resolve(dataframe, 'Connection') is None:
        dataframe = dataframe.withColumn('Connection', F.lit('Uer'))
    if _resolve(dataframe, 'YCode') is None:
        ycode = _resolve(dataframe, 'YCode__c')
        dataframe = dataframe.withColumn('YCode', F.col(ycode) if ycode else F.lit(None).cast('string'))
    profile_name = _resolve(dataframe, 'ProfileName')
    if profile_name:
        dataframe = dataframe.filter(~F.coalesce(F.col(profile_name), F.lit('')).contains('Community'))
        dataframe = dataframe.filter(~F.coalesce(F.col(profile_name), F.lit('')).contains('Partner'))
    return dataframe

def apply_account_rules(dataframe):
    managers = user_lookup()
    for source_column, output_column in (
        ('Primary_Portfolio_Manager__c', 'Primary_Portfolio_Manager_FullName'),
        ('Secondary_Portfolio_Manager__c', 'Secondary_Portfolio_Manager_FullName'),
    ):
        account_column = _resolve(dataframe, source_column)
        if account_column:
            lookup = managers.select(
                F.col('__user_id').alias(f'__{output_column}_key'),
                F.col('__user_name').alias(output_column))
            dataframe = (dataframe.join(lookup, F.col(account_column) == F.col(f'__{output_column}_key'), 'left')
                                  .drop(f'__{output_column}_key'))
    customer_group = _resolve(dataframe, 'SAP_Cust_Group_1__c')
    if customer_group:
        dataframe = dataframe.withColumn('NAP_Account', F.when(F.col(customer_group) == 'NA', 'True').otherwise('False'))
    return dataframe

def apply_opportunity_rules(dataframe):
    owner_column = _resolve(dataframe, 'OwnerId')
    if owner_column:
        dataframe = (dataframe.join(user_lookup().select(
                        F.col('__user_id').alias('__owner_key'),
                        F.col('__user_name').alias('FullName'),
                        F.col('__profile_name').alias('ProfileName')),
                    F.col(owner_column) == F.col('__owner_key'), 'left')
                .drop('__owner_key'))
    campaign_id = _resolve(dataframe, 'CampaignId')
    try:
        campaign = active_bronze('SF_Campaign')
        campaign_key = _resolve(campaign, 'Id')
        campaign_name = _resolve(campaign, 'Name')
        if campaign_id and campaign_key and campaign_name:
            dataframe = (dataframe.join(campaign.select(
                            F.col(campaign_key).alias('__campaign_key'),
                            F.col(campaign_name).alias('CampaignName')),
                        F.col(campaign_id) == F.col('__campaign_key'), 'left')
                    .drop('__campaign_key'))
    except Exception as exc:
        print(f'Campaign enrichment skipped: {exc}')
    return dataframe

def apply_s735_rules(dataframe):
    for column_name in ('SALESREPTRDESC', 'SALESDIVISIONTRDES',
                        'SALESDIVISIONCREDITDE', 'SALESDIVISIONREADES'):
        if _resolve(dataframe, column_name) is None:
            dataframe = dataframe.withColumn(column_name, F.lit(None).cast('string'))
    return dataframe

def apply_workflow_rules(dataframe, table_name):
    if table_name == 'SF_Users':
        return apply_user_rules(dataframe)
    if table_name in ('SF_Accounts', 'SF_ERP_Account'):
        return apply_account_rules(dataframe)
    if table_name in ('SF_Opportunities', 'SF_Opportunities_Summary'):
        return apply_opportunity_rules(dataframe)
    if table_name in ('S735Invoices_Current_REORG', 'S735Invoices_Previous_REORG'):
        return apply_s735_rules(dataframe)
    return dataframe

def project_final_schema(dataframe, final_columns):
    available = {_schema_key(column_name): column_name for column_name in dataframe.columns}
    expressions = []
    missing = []
    for final_column in final_columns:
        source_column = final_column if final_column in dataframe.columns else available.get(_schema_key(final_column))
        if source_column is None:
            expressions.append(F.lit(None).cast('string').alias(final_column))
            missing.append(final_column)
        else:
            expressions.append(F.col(source_column).alias(final_column))
    return dataframe.select(*expressions), missing

for table_name, final_columns in FINAL_COLUMNS.items():
    source_name = FINAL_PRIMARY_SOURCE[table_name]
    table_config = SELLING_TABLE_CONFIGS[source_name]
    dataframe = active_bronze(source_name)
    dataframe = apply_rename_map(dataframe, table_config['rename_map'])
    dataframe = apply_workflow_rules(dataframe, table_name)
    dataframe, missing = project_final_schema(dataframe, final_columns)
    dataframe = (dataframe
                 .withColumn('_PIPELINE_NAME', F.lit(PIPELINE_NAME))
                 .withColumn('_PIPELINE_RUN_ID', F.lit(PIPELINE_RUN_ID))
                 .withColumn('_UPDATED_AT', F.current_timestamp()))
    output_path = f'{SILVER_LH_ABFSS}/{SILVER_SCHEMA}/{table_name}'
    dataframe.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').save(output_path)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.{table_name} USING DELTA LOCATION '{output_path}'")
    row_count = dataframe.count()
    results.append({'table': table_name, 'source': source_name, 'rows': row_count, 'missing_columns': missing})
    print(f'{table_name}: {source_name} -> {len(final_columns)} columns; unresolved: {missing}')

print(f'Silver complete: {len(results)} final schemas written and registered')

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
duration = (datetime.utcnow() - start_time).total_seconds()
print('\n' + '=' * 80)
print('SELLING SYSTEM SILVER — SUMMARY')
print('=' * 80)
print(f'{"Table":<35} {"Source":<32} {"Rows":>12}  Status')
print('-' * 80)
for result in results:
    status = 'OK' if not result['missing_columns'] else f"Missing {len(result['missing_columns'])}"
    print(f'{result["table"]:<35} {result["source"]:<32} {result["rows"]:>12,}  {status}')
print('-' * 80)
print(f'Tables  : {len(results)}')
print(f'Run ID  : {PIPELINE_RUN_ID}')
print(f'Duration: {duration:.1f}s')